## 1. Imports et paramètres
Définir les chemins, les proportions et la graine pour le futur découpage. Exécuter depuis le dossier `PROJET_PNEUMONIA_PYTORCH`.

In [23]:
from pathlib import Path
import random

# Chemins
source = Path("data_original/chest_xray/chest_xray")
destination = Path("data_nouveau_split")

random_state = 42
proportions = {"train": 0.70, "val": 0.15, "test": 0.15}
classes = ("NORMAL", "PNEUMONIA")

## 2. Lister les images originales
Conserver le chemin, la classe et l’ensemble d’origine de chaque image, puis afficher les effectifs. Aucun fichier n’est modifié.

In [24]:
images = []

for ensemble in ("train", "val", "test"):

    for classe in classes:

        dossier = source / ensemble / classe
        nombre = 0

        for fichier in sorted(dossier.iterdir()): #fixer l'ordre (alphabetique)
            if fichier.is_file() and fichier.suffix.lower() in (".jpeg", ".jpg", ".png"): #on verifie qu'on est sur une image
                images.append({"chemin": fichier, "classe": classe, "ancien_split": ensemble})
                nombre += 1

        print(ensemble, classe,":", nombre,"images")

print("Total :", len(images),"images")

train NORMAL : 1341 images
train PNEUMONIA : 3875 images
val NORMAL : 8 images
val PNEUMONIA : 8 images
test NORMAL : 234 images
test PNEUMONIA : 390 images
Total : 5856 images


## 3. Définir les groupes

Le dataset comprend environ 5000 radios pour 3000 personnes: certaines radios sont des radios de la même personne. Comme elles présentes des caractéristiques identiques, il ne serait pas pertinent de séparer plusieurs radios de la même personne dans le train, test ou val car cela constituerait une fuite de données. J'ai donc pris la décision de reprendre toutes les images du dataset et de refaire un split en respectant ce critère. Pour identifier à quelle personne appartient une radio, il faut étudier la nomenclature. Je suis partie de l'hypothèse naturelle suivante : Pour les radios de type normal, le numéro XXXX associé à IM-XXXX désigne une personne en particulier. Pour les radios de type pneumonia, on retrouve "personXXXX" dans le nom. 

In [25]:
def person(fichier, classe):
    nom = fichier.stem  

    if classe == "NORMAL":
        pieces = nom.split("-")
        if pieces[0] == "NORMAL2":
            return "NORMAL::NORMAL2-IM-" + pieces[2]
        else:
            return "NORMAL::IM-" + pieces[1] #car il y a 2 types de noms pour les images normales

    if classe == "PNEUMONIA":
        pieces = nom.split("_")    
        return "PNEUMONIA::" + pieces[0] + "_" + pieces[1]
        
for image in images:
    image["groupe"] = person(image["chemin"], image["classe"])

## 4. Repérer les doublons exacts


Dans le code suivant on réunit les groupes des doublons qui se trouvent chacun dans un groupe différent, c'est une mesure de précaution.

In [26]:
import hashlib

images_uniques = {}
doublons = []

for img in images:
    
    empreinte = hashlib.sha256(img["chemin"].read_bytes()).hexdigest()
    img["empreinte"] = empreinte

    if empreinte not in images_uniques:
        images_uniques[empreinte] = img
    else:
        precedente = images_uniques[empreinte]
        ancien_groupe = img["groupe"]
        nouveau_groupe = precedente["groupe"]
        if ancien_groupe != nouveau_groupe:
            for autre_img in images:
                if autre_img["groupe"] == ancien_groupe:
                    autre_img["groupe"] = nouveau_groupe
        doublons.append(img)

images = list(images_uniques.values())
print("Doublons :", len(doublons))
print("Images conservées :", len(images))

Doublons : 32
Images conservées : 5824


## 5. Construire les groupes définitifs

In [27]:
groupes = set()
for image in images:
    groupes.add(image["groupe"])
print(len(groupes))

for grp in groupes:
    class_grp = set()
    for image in images:
        if image["groupe"] == grp:
            class_grp.add(image["classe"])
    if len(class_grp) > 1:
        raise ValueError("Même groupe contient plusieurs classes : " + grp)


4096


Ce code compte le nombre de groupes total du dataset et renvoie une erreur si l'un des groupes a différentes classes dans ses images (car une personne ne peut pas être saine et atteinte en même temps).

## 6. Répartir les groupes

In [28]:
group_mix = random.Random(random_state).sample(sorted(groupes), len(groupes))

data_train = []
data_val = []
data_test = []

for grp in group_mix:
    images_groupe = [image for image in images if image["groupe"] == grp]
    if len(data_train) < len(images) * proportions["train"]:
        data_train.extend(images_groupe)
    elif len(data_val) < len(images) * proportions["val"]:
        data_val.extend(images_groupe)
    else:
        data_test.extend(images_groupe)

In [29]:
for nom, donnees in (("train", data_train), ("val", data_val), ("test", data_test)):
    print(f"\n{nom} : {len(donnees)} images")
    for classe in classes:
        nombre = sum(image["classe"] == classe for image in donnees)
        print(f"{classe} : {nombre} ({nombre / len(donnees):.1%})")


train : 4077 images
NORMAL : 1128 (27.7%)
PNEUMONIA : 2949 (72.3%)

val : 874 images
NORMAL : 221 (25.3%)
PNEUMONIA : 653 (74.7%)

test : 873 images
NORMAL : 230 (26.3%)
PNEUMONIA : 643 (73.7%)


On vérifie que les proportions de normal et pneumonia sont à peu près les mêmes dans les 3 parties train, val et test.

## 7. Enregistrement des images dans les dossiers de destination

In [30]:
import shutil

destination
destination.mkdir(exist_ok=False)

for nom, donnees in (("train", data_train), ("val", data_val), ("test", data_test)):
    for classe in classes:
        (destination / nom / classe).mkdir(parents=True)
    for image in donnees:
        fichier = image["chemin"]
        nouveau_nom = image["ancien_split"] + "__" + fichier.name
        shutil.copy2(fichier, destination / nom / image["classe"] / nouveau_nom)